<a href="https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/ArishaRamzan-dev/arisha-flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print("Rows:", len(df), "Columns:", len(df.columns))
print(list(df.columns))

Rows: 30000 Columns: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 30, 90, 180, 99999],
    labels=['0-30d', '31-90d', '91-180d', '180d+']
)

bucket_table_1 = df.groupby('staleness_bucket').agg(
    n=('content_id', 'count'),
    avg_clicks_last_30d=('clicks_last_30d', 'mean')
).reset_index()

print(bucket_table_1)

  staleness_bucket      n  avg_clicks_last_30d
0            0-30d  20480             4.209668
1           31-90d    175             4.205714
2          91-180d   9171             6.647694
3            180d+    174             0.574713


/tmp/ipykernel_4678/134079256.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_1 = df.groupby('staleness_bucket').agg(


Verdict: CONFIRMED — content untouched for 180+ days averages only 0.57 clicks/30d, far below every other bucket (4.2–6.6), confirming stale content genuinely underperforms. One caveat: the 91-180d group actually has the highest average clicks, not a steady decline, so staleness isn't a clean gradient — the real cliff is specifically at 180+ days

In [ ]:
df['position_bucket'] = pd.cut(
    df['avg_position'],
    bins=[0, 3, 10, 20, 999],
    labels=['1-3', '4-10', '11-20', '20+']
)

bucket_table_2 = df.groupby('position_bucket').agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean')
).reset_index()

print(bucket_table_2)

  position_bucket      n   avg_ctr
0             1-3   1141  2.714303
1            4-10  11842  0.651045
2           11-20   7273  0.323443
3             20+   8539  0.211333


/tmp/ipykernel_4678/365425540.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket_table_2 = df.groupby('position_bucket').agg(


Verdict: CONFIRMED — CTR drops sharply and consistently with worse position: 2.71 at top positions (1-3) down to 0.21 at 20+, roughly a 13x difference. Unlike the staleness signal, this one is a clean, steady gradient with no surprising bucket.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

median_ctr = df['ctr'].median()

def score_row(row):
    score = 0
    reason = "none"
    if row['days_since_last_update'] > 180:
        score += 50
        reason = "stale_content"
    if row['avg_position'] <= 10 and row['ctr'] < median_ctr:
        score += 30
        if reason == "none":
            reason = "low_ctr_good_position"
    return pd.Series([score, reason])

df[['action_score', 'reason_code']] = df.apply(score_row, axis=1)

def label_action(score):
    if score >= 50:
        return "refresh_now"
    elif score >= 30:
        return "review_soon"
    else:
        return "no_action"

df['action_label'] = df['action_score'].apply(label_action)

ranked = df.sort_values('action_score', ascending=False)

os.makedirs('work/outputs', exist_ok=True)
ranked[['content_id', 'action_score', 'reason_code', 'action_label']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False
)

print(ranked[['content_id', 'action_score', 'reason_code', 'action_label']].head(20))

                 content_id  action_score    reason_code action_label
28250  content_8c66ab2089c0            80  stale_content  refresh_now
3723   content_f488400fca67            80  stale_content  refresh_now
6119   content_cd27391ecd03            80  stale_content  refresh_now
28748  content_ea41fe5cf292            80  stale_content  refresh_now
22980  content_f328d0e4e22b            80  stale_content  refresh_now
24216  content_1b4ec72dafd4            80  stale_content  refresh_now
16417  content_f4b3081037b3            80  stale_content  refresh_now
24454  content_7d58c3076247            80  stale_content  refresh_now
20189  content_277fa742f704            80  stale_content  refresh_now
13316  content_6ca4db5aa791            80  stale_content  refresh_now
7622   content_107776820988            80  stale_content  refresh_now
8631   content_e2b702f4f92b            80  stale_content  refresh_now
8709   content_77834a072fd3            80  stale_content  refresh_now
8675   content_dd2e0

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


1. content_8c66ab2089c0 — refresh_now — stale_content (180+ days, also low CTR for its position) — would be wrong if this page is intentionally evergreen reference content.
2. content_f488400fca67 — refresh_now — stale_content — would be wrong if traffic is seasonal and this is simply an off-season month.
3. content_cd27391ecd03 — refresh_now — stale_content — would be wrong if the content was recently fact-checked outside this system's update log.
4. content_ea41fe5cf292 — refresh_now — stale_content — would be wrong if this ranks for a low-competition query where freshness doesn't matter much.
5. content_f328d0e4e22b — refresh_now — stale_content — would be wrong if this page is a legal/policy page that shouldn't change often.
6. content_1b4ec72dafd4 — refresh_now — stale_content — would be wrong if it's already scheduled for a rewrite outside this tool's tracking.
7. content_f4b3081037b3 — refresh_now — stale_content — would be wrong if it's a cornerstone page whose age is a trust signal, not a weakness.
8. content_7d58c3076247 — refresh_now — stale_content — would be wrong if low CTR here is due to a SERP feature (featured snippet) stealing clicks, not the page itself.
9. content_277fa742f704 — refresh_now — stale_content — would be wrong if this content targets a demographic with different seasonal click patterns.
10. content_6ca4db5aa791 — refresh_now — stale_content — would be wrong if it was updated recently but the update wasn't logged correctly.
11. content_107776820988 — refresh_now — stale_content — would be wrong if the topic itself is naturally low-interest right now industry-wide.
12. content_e2b702f4f92b — refresh_now — stale_content — would be wrong if this page ranks well for a different, unmeasured query.
13. content_77834a072fd3 — refresh_now — stale_content — would be wrong if the content type doesn't benefit from freshness signals (e.g. a static reference table).
14. content_dd2e06be1af4 — refresh_now — stale_content — would be wrong if this is intentionally short-form and doesn't need expansion.
15. content_ab27c30d81f4 — refresh_now — stale_content — would be wrong if it's already outperforming on a metric not captured here (e.g. conversions).
16. content_6557f2b648e8 — refresh_now — stale_content — would be wrong if the low CTR reflects a mismatched but accurate title, not neglect.
17. content_a4da7c7eb188 — refresh_now — stale_content — would be wrong if it's part of a deliberately untouched control set for testing.
18. content_40e140ba2934 — refresh_now — stale_content — would be wrong if the page recently had a URL or tracking change that reset its metrics.
19. content_e2fb3f55bed3 — refresh_now — stale_content — would be wrong if this is a duplicate/near-duplicate of a better-performing page.
20. content_02b0d6e30129 — refresh_now — stale_content — would be wrong if it's genuinely a low-priority page and the score is a false positive from thin data.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: All 20 rows share the exact same score (80) and reason code (stale_content), which means my rule found zero diversity in this dataset's top tier — every top-scoring page hit both conditions (stale AND low-CTR-for-position) at once. I'm least confident in rows like content_f4b3081037b3 and content_a4da7c7eb188, where staleness alone may be misleading if the page is a stable reference/cornerstone piece rather than genuinely neglected — the rule can't currently tell the difference between "old and abandoned" and "old and intentionally stable."

Leakage check: All three signals used — days_since_last_update, ctr, and avg_position — are observed, present-day metrics, not derived from a future window or from the label itself. clicks_last_30d was used only to verify the staleness signal in Section 1, not as a rule input, and it reflects the most recent historical period, not a future one. No column that depends on an outcome after the scoring point was used in the rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.